# TP04: PyTorch in depth, the complete training pipeline

**MSA-DATB09-03 · Neural Networks & Deep Learning Foundations · Session 4 / 8** · Guillaume Saint-Cirgue

*Estimated time: 100 minutes*

In Session 3 you trained a small network with five lines of PyTorch that looked a bit like magic. Today we
open every one of them. This notebook is your **PyTorch reference**: each function is introduced, each
argument is explained, and you write every line yourself. At the end you will have built the complete
pipeline that we will reuse for the rest of the course: data, `Dataset` and `DataLoader`, a model written as
an `nn.Module`, the cross-entropy loss, an optimiser, the training loop, the evaluation loop, and saving the
trained model.

The data: the scikit-learn **digits** dataset, 1797 small grey images of handwritten digits (8x8 pixels,
10 classes). It is tiny, so everything runs in seconds on a laptop CPU.

**How it works.** Each exercise is a cell with a zone to complete:

```python
### START CODE HERE ###

### END CODE HERE ###
```

Write your code between the two markers, then run the cell: the `assert` lines below the zone check your
answer and print `Exercise validated!` when it is correct. If an assert fails, read its message: it tells
you what to look at.

**Run it** on Google Colab (File > Upload notebook) or locally with the course `requirements.txt`.

## Imports

- `numpy`: arrays, used here only to compare with PyTorch.
- `matplotlib`: plots.
- `torch`: PyTorch itself. The main object is the **tensor**, an n-dimensional array (like a NumPy array)
  that can live on a GPU and can record the operations applied to it to compute gradients.
- `torch.nn` (imported as `nn`): the building blocks of neural networks (layers such as `nn.Linear`,
  losses such as `nn.CrossEntropyLoss`, and the base class `nn.Module`).
- `torch.utils.data`: `Dataset` (how to read one example) and `DataLoader` (how to serve mini-batches).
- `sklearn.datasets.load_digits`: the dataset, shipped with scikit-learn (no download).

`torch.manual_seed(0)` fixes the random generator of PyTorch, so that random tensors and random initial
weights are the same every time you run the notebook. `device` is `"cuda"` if a GPU is available, `"cpu"`
otherwise: we will send the model and the data there.

In [1]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import load_digits

FAST_RUN = os.environ.get("FAST_RUN", "0") == "1"   # True: shorter trainings for a quick check
np.random.seed(0)
torch.manual_seed(0)
torch.set_num_threads(2)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "| device:", device)

PyTorch 2.14.1+cpu | device: cpu


# Part A: tensors

A tensor is PyTorch's array. It has three attributes you will check all the time:

| attribute | meaning | example |
|---|---|---|
| `t.shape` | the size of each dimension | `torch.Size([2, 3])`: 2 rows, 3 columns |
| `t.dtype` | the type of the numbers | `torch.float32` (default for decimals), `torch.int64` (default for integers) |
| `t.device` | where it is stored | `cpu` or `cuda:0` |

Most bugs in deep learning code are shape or dtype bugs: printing these three attributes is the first
debugging reflex.

### Exercise 1: create a tensor from a list

`torch.tensor(data)` builds a tensor from a (nested) Python list. If all numbers are integers, the dtype is
`torch.int64`; if at least one is a decimal, it is `torch.float32`. You can force it with the argument
`dtype=torch.float32`.

Create the tensor `t` that contains the matrix
$\begin{pmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \end{pmatrix}$ with dtype `torch.float32`.

> Hint: `torch.tensor([[1., 2., 3.], [4., 5., 6.]])` or `torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)`.

In [3]:
### START CODE HERE ###

t = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)

### END CODE HERE ###

print(t, t.shape, t.dtype, t.device)
assert isinstance(t, torch.Tensor), "t must be a torch.Tensor"
assert t.shape == (2, 3), f"t should have shape (2, 3), not {tuple(t.shape)}"
assert t.dtype == torch.float32, f"t should be float32, not {t.dtype}"
assert t[1, 2].item() == 6.0, "the last element should be 6"
print("Exercise validated!")

tensor([[1., 2., 3.],
        [4., 5., 6.]]) torch.Size([2, 3]) torch.float32 cpu
Exercise validated!


### Exercise 2: creation functions

You rarely type tensors by hand. The usual constructors take the shape as arguments:

- `torch.zeros(3, 4)`: a 3x4 tensor of zeros (`torch.ones` for ones);
- `torch.arange(12)`: the integers 0, 1, ..., 11 (like `range`), dtype `int64`;
- `t.reshape(3, 4)`: the same numbers arranged in a new shape (the total number of elements must not change;
  `-1` means "deduce this size", e.g. `reshape(3, -1)`);
- `torch.randn(1000)`: 1000 numbers drawn from the standard normal distribution $\mathcal{N}(0, 1)$
  (`torch.rand` draws uniformly in $[0, 1)$).

Create `Z` (zeros, shape `(3, 4)`), `R` (the integers 0 to 11 in a `(3, 4)` tensor, row by row) and `G`
(1000 standard normal numbers).

In [10]:
torch.manual_seed(0)

### START CODE HERE ###
Z = torch.zeros(3, 4)
R = torch.arange(12).reshape(3, 4)
G = torch.randn(1000)
### END CODE HERE ###

print(R)
assert Z.shape == (3, 4) and Z.sum().item() == 0, "Z must be a (3, 4) tensor of zeros"
assert R.shape == (3, 4), f"R should have shape (3, 4), not {tuple(R.shape)}"
assert R[0, 0].item() == 0 and R[1, 0].item() == 4 and R[2, 3].item() == 11, "R must contain 0 to 11 row by row"
assert G.shape == (1000,), "G must contain 1000 numbers"
assert abs(G.mean().item()) < 0.15 and abs(G.std().item() - 1) < 0.15, "G should look like N(0, 1): mean near 0, std near 1"
print("Exercise validated!")

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
Exercise validated!


### Exercise 3: indexing and `.item()`

Indexing works as in NumPy: `R[i, j]` is one element, `R[1]` (or `R[1, :]`) is row 1, `R[:, 2]` is column 2,
`R[-1, -1]` is the last element. The result is still a tensor, even for a single number: `.item()`
converts a one-element tensor into a plain Python number (useful to print or store a loss).

From `R`, extract `row1` (row index 1, a tensor), `col2` (column index 2, a tensor) and `last` (the last
element as a Python `int`).

In [19]:
### START CODE HERE ###
row1 = R[1,]
col2 = R[:, 2]
last = R[-1, -1].item()
### END CODE HERE ###

assert torch.equal(row1, torch.tensor([4, 5, 6, 7])), f"row1 should be [4, 5, 6, 7], got {row1}"
assert torch.equal(col2, torch.tensor([2, 6, 10])), f"col2 should be [2, 6, 10], got {col2}"
assert isinstance(last, int) and last == 11, "last should be the Python int 11 (use .item())"
print("Exercise validated!")

Exercise validated!


### Exercise 4: operations, element-wise and matrix product

- `+ - * /` and `**` act **element-wise** (both tensors must have compatible shapes).
- `@` is the **matrix product**: `(n, k) @ (k, p)` gives `(n, p)`. It is the operation of every dense layer.
- `A.T` is the transpose.
- Reductions take a `dim` argument: `A.sum(dim=0)` sums **along** dimension 0, i.e. over the rows, so one
  result per column; `A.mean(dim=1)` gives one mean per row. Without `dim`, the reduction is over all
  elements.

Let `A = R.float()` (`.float()` converts to float32: `mean` refuses integer tensors). Compute
`S = 2A + 1` (element-wise), `P = A A^T` (shape `(3, 3)`), `col_sums` (one sum per column) and `row_means`
(one mean per row).

In [34]:
A = R.float()
### START CODE HERE ###
S = 2*A + 1
P = A@A.T

col_sums = A.sum(dim=0)
row_means = A.mean(dim=1)

print(S)
print(P)

### END CODE HERE ###

assert S.shape == (3, 4) and S[2, 3].item() == 23, "S = 2A + 1 element-wise"
assert P.shape == (3, 3), f"P should be (3, 3), not {tuple(P.shape) if P is not None else None}"
assert P[0, 0].item() == 14 and P[1, 2].item() == 214, "P = A @ A.T"
assert torch.equal(col_sums, torch.tensor([12., 15., 18., 21.])), "col_sums: sum over the rows (dim=0)"
assert torch.equal(row_means, torch.tensor([1.5, 5.5, 9.5])), "row_means: mean over the columns (dim=1)"
print("Exercise validated!")

tensor([[ 1.,  3.,  5.,  7.],
        [ 9., 11., 13., 15.],
        [17., 19., 21., 23.]])
tensor([[ 14.,  38.,  62.],
        [ 38., 126., 214.],
        [ 62., 214., 366.]])
Exercise validated!


### Exercise 5: broadcasting

When the shapes differ, PyTorch (like NumPy) **broadcasts**: it compares the shapes from the right, and a
dimension of size 1 (or a missing dimension) is stretched to match the other tensor. Examples:

- `(3, 4) - (4,)`: the vector is subtracted from **every row**;
- `(3, 4) - (3, 1)`: the column is subtracted from **every column**;
- `(3, 4) - (3,)`: error, because the last dimensions (4 and 3) do not match.

`A.mean(dim=1)` has shape `(3,)`; `A.mean(dim=1, keepdim=True)` keeps the reduced dimension and has
shape `(3, 1)`, which is what we need to broadcast over the rows.

Compute `A_col` (each column of `A` minus the mean of that column) and `A_row` (each row of `A` minus the
mean of that row). You will use the first one to normalise the features of a dataset.

In [ ]:
### START CODE HERE ###

### END CODE HERE ###

assert A_col.shape == (3, 4) and A_row.shape == (3, 4), "the results keep the shape of A"
assert torch.allclose(A_col.mean(dim=0), torch.zeros(4)), "every column of A_col should have mean 0"
assert torch.allclose(A_row.mean(dim=1), torch.zeros(3)), "every row of A_row should have mean 0 (use keepdim=True)"
print("Exercise validated!")

### Exercise 6: the NumPy bridge

- `torch.from_numpy(arr)` turns a NumPy array into a tensor **without copying** (both share the same memory).
  NumPy decimals are `float64`, so the tensor is `torch.float64`.
- Neural networks use `float32` (half the memory, faster): convert with `.float()`.
- `t.numpy()` goes back to NumPy (for a CPU tensor that does not require gradients).

From `arr`, build `t64` (with `from_numpy`), `t32` (its float32 version) and `back` (`t32` converted back to
NumPy).

In [ ]:
arr = np.array([[0.5, 1.5], [2.5, 3.5]])
### START CODE HERE ###

### END CODE HERE ###

assert t64.dtype == torch.float64, "from_numpy keeps NumPy's float64"
assert t32.dtype == torch.float32, "use .float() to convert to float32"
assert isinstance(back, np.ndarray) and back.dtype == np.float32, "back should be a float32 NumPy array"
assert np.allclose(back, arr), "the values must not change"
print("Exercise validated!")

# Part B: autograd

If a tensor is created with `requires_grad=True`, PyTorch **records** every operation that uses it and
builds the computation graph (each result has a `grad_fn` attribute: the function to call during the
backward pass). Then `y.backward()` applies the chain rule backwards through this graph, exactly as we did by
hand in Session 3, and **adds** $\partial y / \partial x$ into `x.grad` for every leaf tensor `x` that requires
gradients. `backward()` is called on a single number (a scalar), typically the loss.

### Exercise 7: the derivative of a function

Let $f(x) = x^3 - 2x$. By hand: $f'(x) = 3x^2 - 2$, so $f'(3) = 25$. Check it with autograd: create `x`, a
float tensor equal to 3 with `requires_grad=True`, compute `f`, call `f.backward()`, and store the
derivative in `grad` (a Python number).

> Hint: `x = torch.tensor(3.0, requires_grad=True)`. The derivative is in `x.grad` after the backward pass.

In [ ]:
### START CODE HERE ###

### END CODE HERE ###

print("f =", f.item(), "| grad_fn:", f.grad_fn, "| df/dx =", grad)
assert abs(f.item() - 21) < 1e-6, "f(3) = 27 - 6 = 21"
assert abs(grad - 25) < 1e-5, "df/dx at x = 3 should be 25"
print("Exercise validated!")

### Exercise 8: the gradient of a sigmoid neuron (Session 2 again)

One example $x = (2, 1)$, label $y = 1$, parameters $w = (0.5, -1)$, $b = 0.5$. The neuron computes
$a = \sigma(w^\top x + b)$ and the log loss is $\mathcal{L} = -[y \log a + (1 - y)\log(1 - a)]$. In Session 2 we
derived $\partial \mathcal{L} / \partial w = (a - y)\,x$ and $\partial \mathcal{L} / \partial b = a - y$.

`w` and `b` are created for you with `requires_grad=True`. Compute `a` (use `torch.sigmoid`), the `loss`
(use `torch.log`), and call `loss.backward()`. The asserts compare `w.grad` and `b.grad` with the formula.

In [ ]:
w = torch.tensor([0.5, -1.0], requires_grad=True)
b = torch.tensor(0.5, requires_grad=True)
x = torch.tensor([2.0, 1.0])
y = torch.tensor(1.0)
### START CODE HERE ###

### END CODE HERE ###

print(f"a = {a.item():.4f}, loss = {loss.item():.4f}, w.grad = {w.grad}, b.grad = {b.grad}")
assert w.grad is not None, "call loss.backward() to fill w.grad"
assert abs(a.item() - 0.6225) < 1e-3, "a = sigmoid(0.5 * 2 - 1 * 1 + 0.5) = sigmoid(0.5)"
assert torch.allclose(w.grad, (a.detach() - y) * x), "w.grad should equal (a - y) x"
assert torch.allclose(b.grad, a.detach() - y), "b.grad should equal a - y"
print("Exercise validated! Autograd computes exactly the formula of Session 2.")

### Exercise 9: gradients accumulate

`backward()` **adds** the new gradient to what is already in `.grad`; it does not overwrite it. This is
useful in some advanced cases, but in a training loop it means we must **reset** the gradients before each
backward pass, otherwise the gradients of all previous batches pile up. `w.grad.zero_()` sets them to zero
in place (the trailing underscore means "in place" in PyTorch). In the training loop,
`optimizer.zero_grad()` does the same for all the parameters.

With $g(w) = (3w)^2 = 9w^2$, $g'(w) = 18w = 18$ at $w = 1$. Run the forward and backward pass **twice** in a
loop, then store `w.grad.item()` in `after_two`. Then reset the gradient and store `w.grad.item()` in
`after_zero`.

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
### START CODE HERE ###

### END CODE HERE ###

print("after two backward passes:", after_two, "| after zero_():", after_zero)
assert after_two == 36, "two backward passes of 18 each should give 36: the gradients add up"
assert after_zero == 0, "after w.grad.zero_() the gradient is 0"
print("Exercise validated!")

### Exercise 10: gradient descent by hand with `torch.no_grad()`

We minimise $h(w) = (w - 3)^2$, whose minimum is at $w = 3$, starting from $w = 0$ with the update
$w \leftarrow w - \eta\, h'(w)$ and $\eta = 0.1$. Each iteration:

1. forward: compute `h` from `w`;
2. backward: `h.backward()` fills `w.grad`;
3. update **inside** `with torch.no_grad():` (the update is not part of the model: we do not want autograd to
   record it, and PyTorch refuses an in-place change of a leaf tensor that requires gradients otherwise),
   with `w -= lr * w.grad`;
4. reset: `w.grad.zero_()`.

This is exactly what `optimizer.step()` and `optimizer.zero_grad()` will do for us in Part E. Related tool:
`t.detach()` returns the same values without the graph (a tensor that does not require gradients), handy to
store or plot a value.

Write the loop for 50 iterations.

In [ ]:
w = torch.tensor(0.0, requires_grad=True)
lr = 0.1
### START CODE HERE ###

### END CODE HERE ###

print("w after 50 steps:", round(w.item(), 5))
assert w.requires_grad, "w must still require gradients (update it inside torch.no_grad())"
assert abs(w.item() - 3) < 1e-3, "after 50 steps w should be very close to 3"
print("Exercise validated!")

# Part C: the data, `Dataset` and `DataLoader`

### Exercise 11: load the digits as tensors

`load_digits()` returns an object with `digits.data`, a NumPy array of shape `(1797, 64)` (each 8x8 image is
flattened into 64 pixels, values from 0 to 16), and `digits.target`, the labels 0 to 9, shape `(1797,)`.

Build:
- `X`: a float32 tensor of the pixels divided by 16 (so that the inputs lie in $[0, 1]$; neural networks
  train much better on small, normalised inputs);
- `y`: an int64 tensor of the labels. `nn.CrossEntropyLoss` requires **integer class indices** (dtype
  `torch.int64`, also called `torch.long`), not floats and not one-hot vectors.

> Hint: `torch.tensor(array, dtype=torch.float32)`.

In [ ]:
digits = load_digits()
print(digits.data.shape, digits.data.dtype, digits.target.shape)
### START CODE HERE ###

### END CODE HERE ###

assert X.shape == (1797, 64) and X.dtype == torch.float32, "X: float32 tensor of shape (1797, 64)"
assert X.max().item() == 1.0 and X.min().item() == 0.0, "divide the pixels by 16 to get values in [0, 1]"
assert y.shape == (1797,) and y.dtype == torch.int64, "y: int64 tensor of shape (1797,)"
print("Exercise validated!")

Helper (run this cell, no need to modify it): a look at the first images. `reshape(8, 8)` puts the 64
pixels back into a square.

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(10, 1.4))
for k, ax in enumerate(axes):
    ax.imshow(X[k].reshape(8, 8), cmap="gray_r")
    ax.set_title(int(y[k]))
    ax.axis("off")
plt.show()

### Exercise 12: train, validation and test sets

We split the 1797 images into three disjoint sets:

| set | size | used for |
|---|---|---|
| train | 1197 | computing the gradients and updating the weights |
| validation | 300 | choosing (number of epochs, learning rate, architecture), watching overfitting |
| test | 300 | reporting the final performance, **once**, at the very end |

`torch.randperm(n, generator=gen)` returns a random permutation of `0, ..., n-1` (the generator with a fixed
seed makes it reproducible). Indexing a tensor with a tensor of indices, `X[idx]`, selects those rows.

Using `perm`, create `train_idx` (the first 1197 indices), `val_idx` (the next 300) and `test_idx` (the last
300), then `X_train, y_train, X_val, y_val, X_test, y_test`.

In [ ]:
gen = torch.Generator().manual_seed(0)
perm = torch.randperm(len(X), generator=gen)
### START CODE HERE ###

### END CODE HERE ###

assert len(X_train) == 1197 and len(X_val) == 300 and len(X_test) == 300, "sizes should be 1197 / 300 / 300"
assert len(set(train_idx.tolist()) | set(val_idx.tolist()) | set(test_idx.tolist())) == 1797, "the three sets must not overlap"
assert torch.equal(X_val[0], X[perm[1197]]) and y_test[-1] == y[perm[-1]], "use the order given by perm"
print("Exercise validated!")

### Exercise 13: a `Dataset` class

A PyTorch `Dataset` answers two questions:

- `__len__(self)`: how many examples are there? (called by `len(ds)`);
- `__getitem__(self, i)`: what is example number `i`? (called by `ds[i]`), returned as a pair `(x, y)`.

`__init__` stores what is needed. For images on disk, `__getitem__` would open the file number `i`; here
the data are already in memory, so it simply indexes the tensors. (PyTorch also has a ready-made
`TensorDataset(X, y)` that does exactly this: we write it once to see that there is no magic.)

Complete the three methods.

In [ ]:
class DigitsDataset(Dataset):
    def __init__(self, X, y):
        ### START CODE HERE ###
        pass
        ### END CODE HERE ###

    def __len__(self):
        ### START CODE HERE ###
        n = None
        ### END CODE HERE ###
        return n

    def __getitem__(self, i):
        ### START CODE HERE ###
        pair = None
        ### END CODE HERE ###
        return pair


train_ds = DigitsDataset(X_train, y_train)
val_ds = DigitsDataset(X_val, y_val)
test_ds = DigitsDataset(X_test, y_test)
assert len(train_ds) == 1197, "len(train_ds) should be 1197"
x0, y0 = train_ds[0]
assert x0.shape == (64,) and y0.shape == (), "ds[i] returns one image (64,) and one label (a 0-d tensor)"
assert torch.equal(x0, X_train[0]) and y0 == y_train[0], "ds[i] must return the i-th example"
print("Exercise validated!")

### Exercise 14: a `DataLoader`

The `DataLoader` wraps a dataset and serves **mini-batches**: it takes `batch_size` examples, stacks them
into one tensor, and hands them over one batch at a time in a `for` loop. Arguments:

- `batch_size=32`: number of examples per batch (the last batch can be smaller);
- `shuffle=True`: a new random order at every epoch, for the training set (the gradient steps then do not
  always see the examples in the same order). For validation and test, `shuffle=False`: order does not matter
  and a fixed order makes results reproducible.

One pass over the loader is one **epoch**, made of `len(loader)` = $\lceil N / B \rceil$ batches, so as many
gradient steps. `next(iter(loader))` takes the first batch (useful to inspect it).

Create `train_loader` (batch size 32, shuffled), `val_loader` and `test_loader` (batch size 256, not
shuffled), then take one training batch `xb, yb`.

In [ ]:
torch.manual_seed(0)
### START CODE HERE ###

### END CODE HERE ###

print("one batch:", xb.shape, xb.dtype, yb.shape, yb.dtype, "| batches per epoch:", len(train_loader))
assert xb.shape == (32, 64) and yb.shape == (32,), "a batch is (32, 64) images and (32,) labels"
assert len(train_loader) == math.ceil(1197 / 32), "38 batches per epoch for 1197 examples"
assert isinstance(train_loader.sampler, torch.utils.data.RandomSampler), "the training loader must shuffle"
assert isinstance(val_loader.sampler, torch.utils.data.SequentialSampler), "do not shuffle the validation set"
print("Exercise validated!")

# Part D: the model, an `nn.Module`

Every PyTorch model is a class that inherits from `nn.Module`:

- `__init__` **declares** the layers. Always start with `super().__init__()` (it sets up the machinery that
  registers the parameters). A layer stored as an attribute (`self.hidden = nn.Linear(...)`) is registered
  automatically: its weights appear in `model.parameters()`.
- `forward(self, x)` **uses** the layers to compute the output from a batch `x`. We never call `forward`
  directly: we write `model(x)`, which calls `forward` (and a few hooks).

`nn.Linear(n_in, n_out)` holds a weight `W` of shape `(n_out, n_in)` and a bias `b` of shape `(n_out,)`, and
computes `x @ W.T + b` for a batch `x` of shape `(batch, n_in)`: this is $Z = W A + b$ of the course, written
with examples in rows. Its weights are initialised randomly (small uniform values).

### Exercise 15: the 64-32-10 network

Write the network of Session 3, adapted to 10 classes: a hidden layer of 32 sigmoid neurons, then an output
layer of 10 neurons that returns the **logits** (raw scores, no softmax: the loss applies it, see Part E).

$$A^{[1]} = \sigma(X W^{[1]\top} + b^{[1]}), \qquad Z^{[2]} = A^{[1]} W^{[2]\top} + b^{[2]}$$

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_in=64, n_hidden=32, n_out=10):
        super().__init__()
        ### START CODE HERE ###
        pass
        ### END CODE HERE ###

    def forward(self, x):
        ### START CODE HERE ###
        logits = None
        ### END CODE HERE ###
        return logits


torch.manual_seed(0)
model = MLP()
print(model)
assert isinstance(model.hidden, nn.Linear) and isinstance(model.out, nn.Linear), "use self.hidden and self.out"
assert model.hidden.weight.shape == (32, 64), "nn.Linear(64, 32).weight has shape (32, 64): (n_out, n_in)"
out = model(xb)
assert out.shape == (32, 10), f"the output should be (batch, 10) logits, not {None if out is None else tuple(out.shape)}"
assert (out < 0).any(), "return the raw logits (no sigmoid or softmax on the output)"
print("Exercise validated!")

### Exercise 16: count the parameters

`model.parameters()` yields every parameter tensor (`W1, b1, W2, b2`); `p.numel()` is the number of elements
of a tensor. `model.named_parameters()` yields `(name, tensor)` pairs, handy to inspect a model:

In [ ]:
for name, p in model.named_parameters():
    print(f"{name:15s} {str(tuple(p.shape)):10s} requires_grad={p.requires_grad}")

By hand: $64 \times 32 + 32 + 32 \times 10 + 10 = 2410$. Compute `n_params` with `parameters()` and `numel()`
in one line.

> Hint: `sum(p.numel() for p in model.parameters())`.

In [ ]:
### START CODE HERE ###

### END CODE HERE ###

assert n_params == 2410, f"expected 2410 parameters, got {n_params}"
print("Exercise validated!")

### Exercise 17: from logits to probabilities and predictions

- `torch.softmax(logits, dim=1)` applies the softmax **along dimension 1** (over the 10 classes of each
  row): each row becomes 10 probabilities that sum to 1.
- `logits.argmax(dim=1)` gives, for each row, the index of the largest logit: the predicted class (the same
  as the argmax of the probabilities, since the softmax keeps the order).

Compute `probs` and `pred` for the batch `xb`, then `acc`, the fraction of correct predictions in the batch
(a Python float). The network is not trained yet: expect about 10 %, i.e. chance.

> Hint: `(pred == yb)` is a boolean tensor; `.float().mean().item()` turns it into a fraction.

In [ ]:
logits = model(xb)
### START CODE HERE ###

### END CODE HERE ###

print("probabilities of the first image:", probs[0].detach().numpy().round(3))
print("predictions:", pred[:10].tolist(), "| accuracy on this batch:", acc)
assert probs.shape == (32, 10) and torch.allclose(probs.sum(dim=1), torch.ones(32)), "each row of probs sums to 1"
assert pred.shape == (32,) and pred.dtype == torch.int64, "pred: one class index per image"
assert isinstance(acc, float) and 0 <= acc <= 0.4, "acc is a float; an untrained model is close to chance"
print("Exercise validated!")

# Part E: loss, optimiser and the training loop

### Exercise 18: the cross-entropy loss

`nn.CrossEntropyLoss()` creates the loss function. It is called as `loss_fn(logits, labels)` with
`logits` of shape `(batch, C)` (raw scores) and `labels` of shape `(batch,)` (int64). Internally it computes
the log-softmax, picks the log-probability of the true class, and averages over the batch:

$$J = -\frac{1}{B}\sum_{i=1}^{B} \log p^{(i)}_{y^{(i)}}$$

Two checks:
1. an untrained network gives probabilities close to $1/10$, so the loss should be close to
   $-\log(1/10) = \log 10 \approx 2.30$. **Always check your initial loss**: a very different value means a bug.
2. compute the same value by hand as `manual`: `torch.log_softmax(logits, dim=1)` gives the log-probabilities,
   `logp[torch.arange(B), yb]` picks, for each row `i`, the column `yb[i]`.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
logits = model(xb)
### START CODE HERE ###

### END CODE HERE ###

print(f"initial loss: {loss0.item():.4f} | log(10) = {math.log(10):.4f} | by hand: {manual.item():.4f}")
assert abs(loss0.item() - math.log(10)) < 0.3, "the initial loss should be close to log(10) = 2.30"
assert torch.allclose(loss0, manual), "manual must equal the value of nn.CrossEntropyLoss"
print("Exercise validated!")

### Exercise 19: one optimisation step

`torch.optim.SGD(model.parameters(), lr=LR)` receives the list of parameters to update. It has two methods:

- `optimizer.zero_grad()`: resets `p.grad` for every parameter (Exercise 9);
- `optimizer.step()`: applies the update rule $p \leftarrow p - \eta\, \nabla_p J$ to every parameter, using the
  gradients stored in `p.grad` (Exercise 10). That is all it does: it does not compute the gradients, the
  backward pass does.

Do one step on the batch `(xb, yb)`: forward (`model(xb)`), loss, `zero_grad`, `backward`, `step`. The assert
checks that the weights moved by exactly $-\eta \times$ gradient.

In [ ]:
LR = 0.5
optimizer = torch.optim.SGD(model.parameters(), lr=LR)
W_before = model.out.weight.detach().clone()   # a copy of the weights, to compare after the step
### START CODE HERE ###

### END CODE HERE ###

assert model.out.weight.grad is not None, "call loss.backward() before optimizer.step()"
expected = W_before - LR * model.out.weight.grad
assert torch.allclose(model.out.weight, expected), "after step(), W should equal W_before - lr * W.grad"
print(f"loss on this batch: before the step {loss.item():.4f}, after {loss_fn(model(xb), yb).item():.4f}")
print("Exercise validated!")

### Exercise 20: the evaluation function

To evaluate, we loop over a loader and accumulate two things: the total loss and the number of correct
predictions. Three rules:

- `model.eval()` switches the model to **evaluation mode**. It changes nothing for our network, but layers
  such as dropout and batch normalisation (Session 6) behave differently in training and in evaluation:
  take the habit now. `model.train()` switches back.
- `with torch.no_grad():` no graph is recorded: faster and less memory, and we do not need gradients to
  evaluate.
- the loss returned by `loss_fn` is a **mean** over the batch; to average over the whole set correctly (the
  last batch may be smaller), multiply it by the batch size before summing, and divide by the total at the
  end.

Complete `evaluate` so that it returns `(mean_loss, accuracy)`, two Python floats.

In [ ]:
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            ### START CODE HERE ###
            pass
            ### END CODE HERE ###
    return total_loss / max(total, 1), correct / max(total, 1)


torch.manual_seed(0)
fresh = MLP().to(device)
val_loss0, val_acc0 = evaluate(fresh, val_loader, loss_fn)
print(f"untrained model: val loss {val_loss0:.4f}, val accuracy {val_acc0:.3f}")
assert abs(val_loss0 - math.log(10)) < 0.3, "untrained: the loss should be close to log(10)"
assert 0 <= val_acc0 <= 0.3, "untrained: the accuracy should be close to 0.1"
assert not fresh.training, "evaluate must put the model in eval mode"
print("Exercise validated!")

### Exercise 21: the training loop

Now the heart of the session. For each epoch:

1. `model.train()`;
2. for each batch `(xb, yb)` of `train_loader`: move it to `device`, then the **five lines**: forward,
   loss, `zero_grad`, `backward`, `step`; add `loss.item() * len(yb)` to a running total to compute the
   mean training loss of the epoch;
3. after the batches: evaluate on the validation set with your `evaluate` function;
4. append the three numbers to `history["train_loss"]`, `history["val_loss"]`, `history["val_acc"]`.

The model, the loss and the optimiser are created for you. Write the loop.

> Hint: the order of the five lines matters: `zero_grad` must come before `backward` (or at the start of the
> iteration), and `step` after `backward`.

In [ ]:
torch.manual_seed(0)
model = MLP().to(device)                                  # .to(device): parameters on the GPU if there is one
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
n_epochs = 30
history = {"train_loss": [], "val_loss": [], "val_acc": []}

for epoch in range(n_epochs):
    ### START CODE HERE ###

    ### END CODE HERE ###
    if epoch % 5 == 4 and history["val_acc"]:
        print(f"epoch {epoch + 1:2d} | train loss {history['train_loss'][-1]:.4f} | "
              f"val loss {history['val_loss'][-1]:.4f} | val acc {history['val_acc'][-1]:.3f}")

assert len(history["train_loss"]) == n_epochs and len(history["val_acc"]) == n_epochs, "one entry per epoch"
assert history["train_loss"][-1] < 0.5 * history["train_loss"][0], "the training loss should at least halve"
assert history["val_acc"][-1] > 0.9, f"validation accuracy should exceed 0.9, got {history['val_acc'][-1]:.3f}"
print("Exercise validated!")

Helper (run this cell, no need to modify it): the learning curves.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot(history["train_loss"], label="train")
ax1.plot(history["val_loss"], label="validation")
ax1.set_xlabel("epoch"); ax1.set_ylabel("cross-entropy"); ax1.legend()
ax2.plot(history["val_acc"], color="C2")
ax2.set_xlabel("epoch"); ax2.set_ylabel("validation accuracy")
plt.tight_layout(); plt.show()

**What we see.** The loss starts at about 2.30 ($\log 10$) and decreases slowly during the first two
epochs (the small initial weights keep the sigmoid outputs close to 0.5 and the logits close to each other),
then it drops quickly.
The validation loss follows the training loss closely: no overfitting yet with 2410 parameters. The
validation accuracy ends above 90 %: a two-layer network written in 15 lines reads handwritten digits.

# Part F: save, reload, and test once

### Exercise 22: save and reload the weights

- `model.state_dict()` is an ordered dictionary `{parameter name: tensor}` (e.g. `"hidden.weight"`): the
  learned numbers, without the code of the class.
- `torch.save(obj, path)` writes it to a file; `torch.load(path)` reads it back.
- To reuse it: create a **new** model of the same class (random weights), then
  `new_model.load_state_dict(torch.load(path))` copies the saved numbers into it.

Save the trained model to `"digits_mlp.pt"`, create `model2 = MLP()`, load the weights into it, put it in
eval mode, and compute `same`, a boolean: are the predictions of `model` and `model2` on `X_test` identical?

> Hint: `torch.equal(a, b)` is True if two tensors have the same shape and values.

In [ ]:
path = "digits_mlp.pt"
### START CODE HERE ###

### END CODE HERE ###

print("keys of the state_dict:", list(model.state_dict().keys()))
assert os.path.exists(path), "save the state_dict with torch.save"
assert torch.equal(model2.hidden.weight, model.hidden.weight), "model2 should have the weights of model"
assert same is True, "the reloaded model must give the same predictions"
print("Exercise validated!")

### Exercise 23: the final test accuracy

We chose everything (architecture, learning rate, number of epochs) by looking at the training and
validation sets only. Now, and only now, we measure the performance on the test set, **once**. This number
estimates the accuracy on new digits. (If you went back to change the model after looking at it, the test
set would become a second validation set and its score would be optimistic.)

Compute `test_loss, test_acc` with your `evaluate` function on `test_loader`.

In [ ]:
### START CODE HERE ###

### END CODE HERE ###

print(f"test loss {test_loss:.4f} | test accuracy {test_acc:.3f}")
assert test_acc > 0.88, f"test accuracy should exceed 0.88, got {test_acc:.3f}"
print("Exercise validated! You wrote the complete PyTorch pipeline.")

# Going further: a reusable `train_one_epoch` / `fit` pair

In the next sessions we will train many models: we do not want to copy the loop each time. We wrap it into
two functions.

### Exercise 24: `train_one_epoch` and `fit`

- `train_one_epoch(model, loader, loss_fn, optimizer)`: one pass over `loader` with the five lines, returns
  the mean training loss (a float).
- `fit(model, train_loader, val_loader, loss_fn, optimizer, n_epochs)`: calls `train_one_epoch` then
  `evaluate` at each epoch and returns the `history` dictionary of Exercise 21.

Then train a fresh `MLP` with `fit` (same settings as Exercise 21) and check that you get the same result.

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    running, total = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        ### START CODE HERE ###
        pass
        ### END CODE HERE ###
    return running / max(total, 1)


def fit(model, train_loader, val_loader, loss_fn, optimizer, n_epochs):
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    for epoch in range(n_epochs):
        ### START CODE HERE ###
        pass
        ### END CODE HERE ###
    return history


torch.manual_seed(0)
model_b = MLP().to(device)
opt_b = torch.optim.SGD(model_b.parameters(), lr=0.5)
hist_b = fit(model_b, train_loader, val_loader, nn.CrossEntropyLoss(), opt_b, n_epochs=30)
print(f"final val accuracy with fit: {hist_b['val_acc'][-1]:.3f} (Exercise 21: {history['val_acc'][-1]:.3f})")
assert len(hist_b["val_acc"]) == 30, "fit must return one entry per epoch"
assert hist_b["val_acc"][-1] > 0.9, "the same training should reach more than 0.9"
print("Exercise validated!")

### Exercise 25: change the optimiser in one line

Thanks to `fit`, trying another optimiser is one line. `torch.optim.Adam(model.parameters(), lr=0.01)`
adapts the step size of each parameter (we will see how in Session 6). Train a fresh `MLP` with Adam for
15 epochs and store the history in `hist_adam`. Compare its curve with SGD's.

In [ ]:
torch.manual_seed(0)
model_adam = MLP().to(device)
### START CODE HERE ###

### END CODE HERE ###

plt.figure(figsize=(5, 3.2))
plt.plot(hist_b["val_acc"], label="SGD, lr = 0.5")
plt.plot(hist_adam["val_acc"], label="Adam, lr = 0.01")
plt.xlabel("epoch"); plt.ylabel("validation accuracy"); plt.legend(); plt.show()
assert len(hist_adam["val_acc"]) == 15, "train for 15 epochs"
assert hist_adam["val_acc"][-1] > 0.9, "Adam should also exceed 0.9"
print("Exercise validated!")